# NER aplicado a textos reales en español: identificación, clasificación y justificación de entidades en sentencias de la Corte Constitucional

**Curso:** Procesamiento de Lenguaje Natural · Unidad 4 — Etiquetado de secuencias y NER
**Institución:** Universidad Tecnológica de Bolívar
**Tipo:** ejemplo resuelto para la actividad individual de NER de la semana
**Referencia:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 17 — Sequence Labeling for Parts of Speech and Named Entities
**Notebooks relacionados:** `ner_spacy.ipynb` (NER preentrenado y reglas), `training_ner.ipynb` (NER propio)
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13

## Contexto y Propósito

Este notebook resuelve, paso a paso, la actividad de la semana: tomar **dos fragmentos reales** de más de 150 palabras, detectar automáticamente sus entidades con **spaCy** (`es_core_news_md`), revisarlas a mano y entregar, por cada fragmento, una tabla con la **entidad, su clasificación (PER, ORG, LOC, DATE, MISC) y una justificación basada en el contexto**. Los fragmentos provienen de dos sentencias públicas de la Corte Constitucional de Colombia, que se descargan desde la relatoría oficial al ejecutar el notebook. Como el modelo de spaCy en español no reconoce fechas, se le añade un **EntityRuler** con patrones de fecha y se mide cuánto tuvo que corregirse la salida automática.

**Objetivo:** mostrar cómo pasar de la salida cruda de una herramienta de NER a una clasificación **justificada y verificable**, y cuantificar la diferencia entre lo que detecta la herramienta y lo que decide una persona que lee el contexto.

## Ejemplos de Aplicación en el Mundo Real

### Gobierno y legal — Indexación de jurisprudencia

- **Problema:** buscar sentencias por autoridad, entidad demandada o fecha de la decisión.
- **Entrada:** texto de las sentencias.
- **Cómo se aplica el NER:**
  1. spaCy detecta organizaciones, personas y lugares; un EntityRuler añade fechas.
  2. Un revisor corrige las entidades mal delimitadas (p. ej. *Sección Cuarta* separada de *Consejo de Estado*).
  3. Las entidades validadas alimentan un índice.
- **Salida:** ficha por sentencia con autoridades, partes y fechas.
- **Decisión:** búsqueda y seguimiento de casos por entidad.

### Periodismo — Verificación de notas sobre decisiones judiciales

- **Problema:** un medio quiere comprobar que sus notas nombran correctamente tribunales, partes y fechas.
- **Entrada:** la nota y la sentencia original.
- **Cómo se aplica el NER:** se extraen las entidades de ambos textos y se comparan; las diferencias (una fecha o un tribunal distinto) se marcan para revisión.
- **Salida:** lista de discrepancias.
- **Decisión:** corregir la nota antes de publicarla.

### Finanzas y servicio al cliente — Riesgo legal de contrapartes

- **Problema:** un banco quiere saber si un cliente corporativo aparece en litigios.
- **Entrada:** sentencias públicas.
- **Cómo se aplica el NER:** se buscan las organizaciones (ORG) y los montos de condenas en los textos.
- **Salida:** alertas por cliente con el monto y la fecha de la decisión.
- **Decisión:** ajustar la evaluación de riesgo del cliente.

## Requisitos Previos

**Instalación** (una sola vez, desde la terminal; en Jupyter antepón `!`):

```bash
pip install spacy pandas requests beautifulsoup4 lxml
python -m spacy download es_core_news_md
```

**Librerías que se importan:**

| Librería | Para qué se usa |
|---|---|
| `spacy`, `spacy.displacy` | NER preentrenado, EntityRuler para fechas y visualización |
| `requests`, `bs4` | Descarga de las sentencias desde la relatoría de la Corte Constitucional |
| `pandas` | Tablas de resultados |
| `re`, `datetime`, `collections` | Extracción del fragmento, fecha de consulta y conteos |

**Fuentes de los fragmentos** (textos públicos de la Corte Constitucional de Colombia):

| Fragmento | Fuente | Enlace |
|---|---|---|
| 1 | Corte Constitucional, Sentencia SU-499 de 2024 (Sala Plena, 28 de noviembre de 2024) | [su499-24.htm](https://www.corteconstitucional.gov.co/relatoria/2024/su499-24.htm) |
| 2 | Corte Constitucional, Sentencia T-390 de 2012 (M. P. Nilson Pinilla Pinilla, 28 de mayo de 2012) | [t-390-12.htm](https://www.corteconstitucional.gov.co/relatoria/2012/t-390-12.htm) |

> ℹ️ El notebook necesita internet para descargar las sentencias la primera vez; luego usa la copia guardada en `data/base_ner/`.

## Marco metodológico: CRISP-ML(Q)

Este notebook sigue **CRISP-ML(Q)** (Studer et al., 2021): seis fases, cada una con sus requisitos, riesgos y controles de calidad (QA). Cada fase se marca con 🔷 y muestra un aspecto **específico** de la actividad.

| Fase | Aspecto específico en esta actividad | Control de calidad (QA) |
|---|---|---|
| 🔷 1. Comprensión del negocio y de los datos | Requisitos de la actividad y equivalencia entre las etiquetas de spaCy y las categorías pedidas | Lista de requisitos verificable |
| 🔷 2. Ingeniería de datos | Descarga de textos reales, extracción de fragmentos ≥ 150 palabras y registro de la fuente | `assert` de longitud, origen y fecha de consulta |
| 🔷 3. Ingeniería del modelo | spaCy `es_core_news_md` + EntityRuler para `DATE`; versión y parámetros registrados | Prueba de los patrones con frases de control |
| 🔷 4. Evaluación | Revisión manual de cada entidad y medida de cuánto acertó la herramienta | Precisión y *recall* de spaCy frente a la revisión manual |
| 🔷 5. Despliegue | Informe: fragmentos, fuentes, tablas *Entidad · Clasificación · Justificación* y reflexión | Toda entidad tiene justificación; ninguna queda como borrador |
| 🔷 6. Monitoreo y mantenimiento | Qué revisar al cambiar de textos, de modelo o de versión | Registro de versiones y lista de verificación |

## Estándares de referencia

Prácticas de este notebook alineadas con estándares de IEEE SA, ISO/IEC y la Open Source Initiative (OSI). Se citan como guía de buenas prácticas, no como certificación de cumplimiento.

| Estándar | Cómo se aplica aquí |
|---|---|
| **IEEE 7001-2021** — Transparencia de sistemas autónomos | Se registran librería, versión, modelo y patrones del EntityRuler; cada clasificación se justifica y se indica si vino de spaCy o de la revisión manual. |
| **IEEE 7003-2024** — Consideraciones sobre sesgo algorítmico | El modelo se entrenó sobre todo con noticias de España; la fase 4 mide cómo falla con nombres de tribunales colombianos. |
| **ISO/IEC 23894:2023** — Guía para la gestión de riesgos de IA | Riesgo documentado: aceptar la salida automática sin revisión produce entidades mal delimitadas y etiquetas erróneas. |
| **IEEE 7000-2021** — Consideraciones éticas en el diseño | Se usan textos públicos con su fuente; los datos personales que aparecen son los publicados por la propia Corte. |
| **OSI · Open Source AI Definition 1.0 (2024)** | spaCy es software libre (MIT); la licencia del modelo se imprime desde sus metadatos en la fase 3. |

## Objetivos de aprendizaje

Al finalizar este notebook podrás:

1. **Extraer** fragmentos de textos reales y documentar su fuente.
2. **Aplicar** spaCy y ampliar sus categorías con un EntityRuler.
3. **Clasificar** cada entidad según su sentido en el fragmento, aunque la herramienta sugiera otra etiqueta.
4. **Justificar** cada clasificación en 1–2 líneas con base en el contexto.
5. **Medir** cuánto acertó la herramienta frente a la revisión manual y redactar una reflexión con evidencia.

---

## 0. Configuración del entorno

In [1]:
import re
import datetime as dt
from collections import Counter
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
import spacy
from spacy import displacy

pd.set_option('display.max_colwidth', 120)
DATA_DIR = Path('data/base_ner')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('spaCy', spacy.__version__)

spaCy 3.8.16


---

## 1. 🔷 Fase 1 — Comprensión del negocio y de los datos

**Aspecto específico:** traducir el enunciado de la actividad en requisitos verificables y decidir cómo se corresponden las etiquetas de la herramienta con las categorías pedidas.

| Requisito de la actividad | Cómo se verifica en el notebook |
|---|---|
| Dos fragmentos distintos de fuentes reales, lenguaje formal, tema claro | Dos sentencias públicas descargadas de la relatoría (fase 2) |
| Mínimo 150 palabras por fragmento | `assert` en la fase 2 |
| Fuente indicada (título y enlace) | Tabla de fuentes con fecha de consulta (fase 2) |
| Herramienta, versión, modelo y parámetros | Registro impreso en la fase 3 |
| Categorías PER, ORG, LOC, DATE, MISC según el contexto | Revisión manual con reetiquetado (fase 4) |
| Justificación de 1–2 líneas por entidad | Tabla final; el notebook avisa si alguna queda como borrador (fase 5) |
| Reflexión ≤ 5 líneas sobre aciertos y ajustes | Calculada con las cifras de la fase 4 (fase 5) |

**Equivalencia de etiquetas:** `es_core_news_md` reconoce `PER`, `ORG`, `LOC` y `MISC`, pero **no fechas**. `DATE` se añade con un EntityRuler. Los montos de dinero, que la actividad no pide, se etiquetan como `MONEY`, una categoría adicional que la actividad permite cuando la herramienta la soporta.

In [2]:
CATEGORIAS = {
    'PER': 'Persona',
    'ORG': 'Organización',
    'LOC': 'Lugar geográfico (ciudad, país, región)',
    'DATE': 'Fecha o período temporal',
    'MISC': 'Otra entidad relevante (eventos, productos, títulos, normas, procesos)',
    'MONEY': 'Monto de dinero (categoría adicional)',
}
MIN_PALABRAS = 150
pd.DataFrame({'Etiqueta': CATEGORIAS.keys(), 'Significado': CATEGORIAS.values(),
              'Origen': ['spaCy', 'spaCy', 'spaCy', 'EntityRuler', 'spaCy', 'EntityRuler']})

,Etiqueta,Significado,Origen
0,PER,Persona,spaCy
1,ORG,Organización,spaCy
2,LOC,"Lugar geográfico (ciudad, país, región)",spaCy
3,DATE,Fecha o período temporal,EntityRuler
4,MISC,"Otra entidad relevante (eventos, productos, títulos, normas, procesos)",spaCy
5,MONEY,Monto de dinero (categoría adicional),EntityRuler


> **Interpretación de la salida:** las cuatro primeras categorías y `MISC` vienen del modelo; `DATE` y `MONEY` las aporta el EntityRuler. Fijar esta tabla antes de ver resultados evita cambiar las reglas de clasificación a mitad del análisis (QA de la fase 1).

---

## 2. 🔷 Fase 2 — Ingeniería de datos

**Aspecto específico:** obtener textos **reales** y trazables. Cada fragmento empieza en una frase verificada del documento y se corta en el primer final de oración después de 150 palabras. Se guarda la fecha de consulta para la referencia bibliográfica.

**Plan B:** si no hay internet, pega cada fragmento, copiado de la página oficial, en `FRAGMENTOS_MANUALES`.

In [3]:
FUENTES = {
    'Fragmento 1': {
        'titulo': 'Corte Constitucional de Colombia. Sentencia SU-499 de 2024 (28 de noviembre de 2024)',
        'url': 'https://www.corteconstitucional.gov.co/relatoria/2024/su499-24.htm',
        'inicio': 'La Corte Constitucional amparó el derecho al debido proceso de la Universidad de Cartagena',
    },
    'Fragmento 2': {
        'titulo': 'Corte Constitucional de Colombia. Sentencia T-390 de 2012 (28 de mayo de 2012), M. P. Nilson Pinilla Pinilla',
        'url': 'https://www.corteconstitucional.gov.co/relatoria/2012/t-390-12.htm',
        'inicio': 'Esclarecido el panorama normativo que determina',
    },
}
FRAGMENTOS_MANUALES = {}   # p. ej. {'Fragmento 1': 'texto copiado de la página oficial…'}
MAX_PALABRAS = 320


def descargar_texto(url: str) -> str:
    """Texto plano de la página (con copia local para no descargarla de nuevo)."""
    cache = DATA_DIR / (Path(url).stem + '.txt')
    if cache.exists():
        return cache.read_text(encoding='utf-8')
    r = requests.get(url, headers={'User-Agent': 'courseNLP-UTB/1.0 (uso academico)'}, timeout=60)
    r.raise_for_status()
    r.encoding = r.apparent_encoding or 'utf-8'
    soup = BeautifulSoup(r.text, 'lxml')
    for s in soup(['script', 'style']):
        s.decompose()
    texto = re.sub(r'\s+', ' ', soup.get_text(' ')).strip()
    cache.write_text(texto, encoding='utf-8')
    return texto


def extraer_fragmento(texto: str, inicio: str, minimo: int = MIN_PALABRAS, maximo: int = MAX_PALABRAS) -> str:
    """Desde la frase de inicio hasta el primer final de oración tras `minimo` palabras."""
    patron = r'\s+'.join(map(re.escape, inicio.split()))
    m = re.search(patron, texto)
    if not m:
        raise ValueError(f'No se encontró el inicio: «{inicio[:60]}…»')
    resto = texto[m.start():]
    palabras = resto.split(' ')
    corte = ' '.join(palabras[:minimo])
    # Primer punto (no precedido de dígito, para no cortar en «6.2.») seguido de mayúscula o cifra
    fin = re.search(r'(?<!\d)\.\s+(?=[A-ZÁÉÍÓÚÑ0-9«“"])', resto[len(corte):])
    fragmento = resto[:len(corte) + fin.start() + 1] if fin else corte
    return ' '.join(fragmento.split(' ')[:maximo]).strip()


fragmentos, registro_fuentes = {}, []
for nombre, f in FUENTES.items():
    if nombre in FRAGMENTOS_MANUALES:
        fragmentos[nombre], origen = FRAGMENTOS_MANUALES[nombre].strip(), 'copiado a mano'
    else:
        fragmentos[nombre], origen = extraer_fragmento(descargar_texto(f['url']), f['inicio']), 'descargado'
    n = len(fragmentos[nombre].split())
    assert n >= MIN_PALABRAS, f'{nombre} tiene {n} palabras (< {MIN_PALABRAS})'   # QA
    registro_fuentes.append({'Fragmento': nombre, 'Fuente': f['titulo'], 'Enlace': f['url'],
                             'Palabras': n, 'Obtenido': origen, 'Consulta': dt.date.today().isoformat()})

pd.DataFrame(registro_fuentes)

,Fragmento,Fuente,Enlace,Palabras,Obtenido,Consulta
0,Fragmento 1,Corte Constitucional de Colombia. Sentencia SU-499 de 2024 (28 de noviembre de 2024),https://www.corteconstitucional.gov.co/relatoria/2024/su499-24.htm,201,descargado,2026-11-03
1,Fragmento 2,"Corte Constitucional de Colombia. Sentencia T-390 de 2012 (28 de mayo de 2012), M. P. Nilson Pinilla Pinilla",https://www.corteconstitucional.gov.co/relatoria/2012/t-390-12.htm,245,descargado,2026-11-03


> **Interpretación de la salida:** cada fragmento supera las 150 palabras (si no, el `assert` detiene el notebook) y queda con su título, enlace y fecha de consulta, que es lo que exige la actividad para citar la fuente. La columna `Obtenido` deja constancia de si el texto se descargó o se copió a mano.

In [4]:
for nombre, texto in fragmentos.items():
    print(f'=== {nombre} ({len(texto.split())} palabras) ===\n{texto}\n')

=== Fragmento 1 (201 palabras) ===
La Corte Constitucional amparó el derecho al debido proceso de la Universidad de Cartagena, vulnerado por la Sección Cuarta del Consejo de Estado al confirmar en segunda instancia mediante providencia del 11 de mayo de 2023, la sentencia de la Sala Fija de Decisión del Tribunal Administrativo de Bolívar proferida en primera instancia el 29 de mayo de 2020 dentro del proceso de nulidad y restablecimiento del derecho adelantado por Ecopetrol contra los actos administrativos mediante los cuales la Universidad de Cartagena le negó la devolución de los dineros pagados por concepto de “estampilla Universidad de Cartagena siempre a la altura de los tiempos”. En consecuencia, dejó sin efectos la sentencia de segunda instancia objeto de la tutela y ordenó a la Sección Cuarta del Consejo de Estado que rehiciera el trámite del recurso de apelación dentro del proceso de nulidad y restablecimiento del derecho , y que, previa incorporación al expediente de la total

> **Interpretación de la salida:** estos son los fragmentos **en su idioma original**, tal como irán en el informe. Comprueba que el texto empieza y termina en oraciones completas y que no arrastra restos de la página (menús, notas al pie).

---

## 3. 🔷 Fase 3 — Ingeniería del modelo

**Aspecto específico:** documentar la herramienta (librería, versión, modelo y parámetros) y ampliar sus categorías. El EntityRuler se coloca **antes** del componente `ner`: así sus fechas y montos tienen prioridad y el modelo no los reetiqueta.

| Patrón | Ejemplo que reconoce |
|---|---|
| día + *de* + mes + *de/del* + año | *11 de mayo de 2023* |
| mes + día + *de* + año | *octubre 7 de 2011* |
| día + *de* + mes | *28 de noviembre* |
| mes + *de* + año | *mayo de 2012* |
| `$` + número | *$22.586.296.153* |

In [5]:
MODELO = 'es_core_news_md'
nlp = spacy.load(MODELO)

MESES = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto',
         'septiembre', 'setiembre', 'octubre', 'noviembre', 'diciembre']
MES = {'LOWER': {'IN': MESES}}
PATRONES = [
    {'label': 'DATE', 'pattern': [{'IS_DIGIT': True}, {'LOWER': 'de'}, MES, {'LOWER': {'IN': ['de', 'del']}}, {'IS_DIGIT': True}]},
    {'label': 'DATE', 'pattern': [MES, {'IS_DIGIT': True}, {'LOWER': 'de'}, {'IS_DIGIT': True}]},
    {'label': 'DATE', 'pattern': [{'IS_DIGIT': True}, {'LOWER': 'de'}, MES]},
    {'label': 'DATE', 'pattern': [MES, {'LOWER': 'de'}, {'IS_DIGIT': True}]},
    {'label': 'MONEY', 'pattern': [{'TEXT': '$'}, {'LIKE_NUM': True}]},
    {'label': 'MONEY', 'pattern': [{'TEXT': {'REGEX': r'^\$[\d.,]+$'}}]},
]
ruler = nlp.add_pipe('entity_ruler', before='ner')
ruler.add_patterns(PATRONES)

# Registro de la herramienta (lo pide la actividad)
registro_herramienta = {
    'Librería': f'spaCy {spacy.__version__}',
    'Modelo': f"{MODELO} {nlp.meta['version']} (licencia: {nlp.meta.get('license', 'ver metadatos')})",
    'Componentes': ', '.join(nlp.pipe_names),
    'Parámetros': f'EntityRuler antes de ner con {len(PATRONES)} patrones (DATE, MONEY); sin umbrales de confianza',
}
for k, v in registro_herramienta.items():
    print(f'{k:12}: {v}')

# QA: frases de control para los patrones
for frase in ['La providencia del 11 de mayo de 2023 fue confirmada.',
              'El fallo de octubre 7 de 2011 se revocó.', 'Debe pagar $22.586.296.153 al demandante.']:
    print(' ', [(e.text, e.label_) for e in nlp(frase).ents])

Librería    : spaCy 3.8.16
Modelo      : es_core_news_md 3.8.0 (licencia: GNU GPL 3.0)
Componentes : tok2vec, morphologizer, parser, attribute_ruler, lemmatizer, entity_ruler, ner
Parámetros  : EntityRuler antes de ner con 6 patrones (DATE, MONEY); sin umbrales de confianza
  [('11 de mayo de 2023', 'DATE')]
  [('octubre 7 de 2011', 'DATE')]
  [('$22.586.296.153', 'MONEY')]


> **Interpretación de la salida:** el registro es lo que la actividad pide declarar cuando se usa una herramienta automática. Las frases de control comprueban que cada patrón funciona antes de aplicarlo a los fragmentos; si alguna fecha no aparece, el patrón tiene un error.

In [6]:
docs = {nombre: nlp(texto) for nombre, texto in fragmentos.items()}
COLORES = {'DATE': '#c7e9c0', 'MONEY': '#fdd0a2'}
for nombre, doc in docs.items():
    print(f'=== {nombre}: {len(doc.ents)} entidades detectadas por la herramienta ===')
    displacy.render(doc, style='ent', jupyter=True, options={'colors': COLORES})

=== Fragmento 1: 12 entidades detectadas por la herramienta ===


=== Fragmento 2: 20 entidades detectadas por la herramienta ===


> **Interpretación de la salida:** esta es la **salida cruda** de la herramienta, antes de cualquier revisión. En textos jurídicos colombianos es común que spaCy parta nombres de tribunales (*Sección Cuarta* | *Consejo de Estado*), marque como lugar o persona palabras en mayúscula al inicio de la oración (*Esclarecido*, *Impugnado*) y confunda órganos judiciales con lugares (*Circuito de Cartagena*). La fase 4 corrige esos casos.

---

## 4. 🔷 Fase 4 — Evaluación: revisión manual y medida de la herramienta

**Aspecto específico:** la actividad exige que la clasificación **corresponda al contexto, aunque la herramienta sugiera otra etiqueta**. Las decisiones de la revisión se escriben como datos, no como ediciones a mano en la tabla, para que queden documentadas y se puedan repetir:

| Ajuste | Cuándo se usa | Ejemplo |
|---|---|---|
| `eliminar` | La herramienta marcó algo que no es una entidad | *Esclarecido* (verbo al inicio de oración) |
| `fusionar` | Una entidad quedó partida o mal delimitada | *Sección Cuarta del Consejo de Estado* |
| `reetiquetar` | El tipo no corresponde al contexto | — |
| `agregar` | La herramienta no detectó una entidad relevante | *nulidad y restablecimiento del derecho* (MISC) |

Criterios aplicados en la revisión:
- Un nombre de lugar **dentro** del nombre de una institución (*Universidad de Cartagena*, *Tribunal Administrativo de Bolívar*) forma parte de la `ORG`; no se cuenta como `LOC` aparte.
- Los cargos sin nombre propio (*Jefe de la Oficina Jurídica*) no son `PER`; se conserva la organización.
- Las referencias genéricas (*la Sala*, *ese Instituto*) no se cuentan como entidades nuevas.

In [7]:
AJUSTES = {
    'Fragmento 1': {
        'eliminar': [],
        'fusionar': [('Sección Cuarta del Consejo de Estado', 'ORG')],
        'reetiquetar': {},
        'agregar': [('nulidad y restablecimiento del derecho', 'MISC')],
    },
    'Fragmento 2': {
        'eliminar': ['Esclarecido', 'Sala', 'Instituto', 'El fallo de primera instancia', 'Impugnado',
                     'Jefe de la Oficina Jurídica de INVÍAS'],
        'fusionar': [('Juzgados Quinto y Sexto Civiles de Circuito de Cartagena', 'ORG'),
                     ('Juzgado Quinto Civil del Circuito de Cartagena', 'ORG'),
                     ('Juez Quinto Civil del Circuito de Cartagena', 'ORG'),
                     ('Tribunal Superior de Cartagena, Sala Civil Familia', 'ORG'),
                     ('Corte Suprema de Justicia, Sala de Casación Civil', 'ORG')],
        'reetiquetar': {},
        'agregar': [('INVÍAS', 'ORG'), ('0075 de 2001', 'MISC')],
    },
}


def buscar(texto: str, frase: str) -> list:
    """Posiciones (inicio, fin) de todas las apariciones de una frase, tolerando espacios distintos."""
    patron = r'\s+'.join(map(re.escape, frase.split()))
    return [(m.start(), m.end()) for m in re.finditer(patron, texto)]


def revisar(nombre: str, doc) -> tuple:
    """Aplica los ajustes manuales a la salida de spaCy. Devuelve (automáticas, finales, avisos)."""
    texto, aj, avisos = doc.text, AJUSTES.get(nombre, {}), []
    auto = [{'inicio': e.start_char, 'fin': e.end_char, 'texto': e.text, 'etiqueta_spacy': e.label_} for e in doc.ents]
    finales = [dict(e, etiqueta=e['etiqueta_spacy'], origen='spaCy') for e in auto
               if e['texto'] not in aj.get('eliminar', [])]
    for frase, etiqueta in aj.get('fusionar', []) + aj.get('agregar', []):
        posiciones = buscar(texto, frase)
        if not posiciones:
            avisos.append(f'«{frase}» no aparece en {nombre}: revisa el ajuste'); continue
        for ini, fin in posiciones:
            previas = [e for e in finales if e['inicio'] < fin and ini < e['fin']]
            if any(e['inicio'] == ini and e['fin'] == fin and e['etiqueta'] == etiqueta for e in previas):
                continue   # ya estaba bien
            finales = [e for e in finales if e not in previas]
            origen = ('manual: fusión de ' + ' + '.join(f"«{e['texto']}» ({e['etiqueta_spacy']})" for e in previas)
                      if previas else 'manual: agregada (spaCy no la detectó)')
            finales.append({'inicio': ini, 'fin': fin, 'texto': texto[ini:fin], 'etiqueta_spacy': '—',
                            'etiqueta': etiqueta, 'origen': origen})
    for e in finales:
        nueva = aj.get('reetiquetar', {}).get(e['texto'])
        if nueva and nueva != e['etiqueta']:
            e['origen'] = f"manual: reetiquetada {e['etiqueta']} → {nueva}"; e['etiqueta'] = nueva
    for t in aj.get('eliminar', []):
        if t not in {e['texto'] for e in auto}:
            avisos.append(f'«{t}» no estaba en la salida de spaCy de {nombre}')
    return auto, sorted(finales, key=lambda e: e['inicio']), avisos


revision = {}
for nombre, doc in docs.items():
    auto, finales, avisos = revisar(nombre, doc)
    revision[nombre] = (auto, finales)
    print(f'{nombre}: {len(auto)} entidades de spaCy → {len(finales)} tras la revisión')
    for a in avisos:
        print('  ⚠️', a)

Fragmento 1: 12 entidades de spaCy → 12 tras la revisión
Fragmento 2: 20 entidades de spaCy → 13 tras la revisión
  ⚠️ «Juez Quinto Civil del Circuito de Cartagena» no aparece en Fragmento 2: revisa el ajuste
  ⚠️ «Corte Suprema de Justicia, Sala de Casación Civil» no aparece en Fragmento 2: revisa el ajuste
  ⚠️ «Impugnado» no estaba en la salida de spaCy de Fragmento 2


> **Interpretación de la salida:** si aparece un aviso, el texto descargado difiere del que se usó para escribir los ajustes (por ejemplo, un espacio o una palabra distinta): ajusta la frase en `AJUSTES` para que coincida con el fragmento. Con textos propios, este diccionario es exactamente lo que cada estudiante debe llenar.

In [8]:
def comparar(auto: list, finales: list) -> dict:
    """Precisión y recall de spaCy tomando la revisión manual como referencia (texto + etiqueta exactos)."""
    a = {(e['texto'], e['etiqueta_spacy']) for e in auto}
    f = {(e['texto'], e['etiqueta']) for e in finales}
    vp = len(a & f)
    prec = vp / len(a) if a else 0.0
    rec = vp / len(f) if f else 0.0
    return {'Detectadas por spaCy': len(a), 'Tras revisión': len(f), 'Correctas sin cambios': vp,
            'Precisión spaCy': round(prec, 2), 'Recall spaCy': round(rec, 2),
            'F1 spaCy': round(2 * prec * rec / (prec + rec), 2) if prec + rec else 0.0}


medidas = pd.DataFrame({n: comparar(*revision[n]) for n in revision}).T
enteras = ['Detectadas por spaCy', 'Tras revisión', 'Correctas sin cambios']
medidas[enteras] = medidas[enteras].astype(int)
medidas

,Detectadas por spaCy,Tras revisión,Correctas sin cambios,Precisión spaCy,Recall spaCy,F1 spaCy
Fragmento 1,9,9,7,0.78,0.78,0.78
Fragmento 2,18,13,8,0.44,0.62,0.52


> **Interpretación de la salida:** se cuentan entidades distintas (texto + etiqueta). La **precisión** indica qué parte de lo que marcó spaCy era correcta tal cual; el **recall**, qué parte de las entidades finales encontró sin ayuda. Una precisión baja significa que hubo que eliminar o corregir mucho; un recall bajo, que hubo que agregar entidades. Estas cifras sostienen la reflexión de la fase 5.

In [9]:
# ¿Qué tipo de ajuste se hizo, por fragmento?
tipos_ajuste = {n: Counter(e['origen'].split(':')[0] + (': fusión' if 'fusión' in e['origen'] else
                                                      ': agregada' if 'agregada' in e['origen'] else '')
                           for e in revision[n][1]) for n in revision}
for n in revision:
    auto, finales = revision[n]
    # Descartadas: detecciones de spaCy que no se solapan con ninguna entidad final
    eliminadas = sum(not any(a['inicio'] < f['fin'] and f['inicio'] < a['fin'] for f in finales) for a in auto)
    print(n, dict(tipos_ajuste[n]), f'· detecciones descartadas: {eliminadas}')

Fragmento 1 {'spaCy': 8, 'manual: fusión': 2, 'manual: agregada': 2} · detecciones descartadas: 0
Fragmento 2 {'manual: agregada': 2, 'manual: fusión': 3, 'spaCy': 8} · detecciones descartadas: 4


> **Interpretación de la salida:** muestra cuántas entidades finales se aceptaron tal como las dio spaCy y cuántas exigieron fusión o se agregaron a mano, además de cuántas detecciones se descartaron. Es la forma cuantitativa de responder si la herramienta «detectó correctamente todas las entidades».

---

## 5. 🔷 Fase 5 — Despliegue: el informe

**Aspecto específico:** el producto de la actividad es un informe. Cada entidad final lleva una **justificación de 1–2 líneas** escrita a partir del contexto. Las justificaciones se guardan en un diccionario; si alguna entidad no tiene una, el notebook genera un **borrador** con su contexto y lo marca con ⚠️ para completarlo.

In [10]:
JUSTIFICACIONES = {
    'Fragmento 1': {
        'Corte Constitucional': 'Es el tribunal que dicta la sentencia y ampara el derecho; actúa como institución, no como lugar ni persona.',
        'Universidad de Cartagena': 'Es la institución de educación superior cuyo derecho al debido proceso se ampara; «Cartagena» forma parte de su nombre, no se usa como lugar.',
        'Sección Cuarta del Consejo de Estado': 'Es la sección del máximo tribunal de lo contencioso administrativo que dictó la providencia cuestionada; spaCy la partió en dos y se unió.',
        'Consejo de Estado': 'Es el máximo tribunal de lo contencioso administrativo de Colombia; en el fragmento actúa como autoridad judicial.',
        '11 de mayo de 2023': 'Es la fecha de la providencia de segunda instancia que confirmó la sentencia; marca un momento del proceso.',
        'Sala Fija de Decisión del Tribunal Administrativo de Bolívar': 'Es el órgano judicial que dictó la sentencia de primera instancia; «Bolívar» identifica su jurisdicción dentro del nombre.',
        '29 de mayo de 2020': 'Es la fecha en que se dictó la sentencia de primera instancia.',
        'Ecopetrol': 'Es la empresa que promovió el proceso contra los actos de la Universidad; actúa como parte, es una organización.',
        'nulidad y restablecimiento del derecho': 'Nombra un medio de control judicial (tipo de proceso), no una persona ni una institución; por eso es MISC.',
    },
    'Fragmento 2': {
        'INVÍAS': 'Es el Instituto Nacional de Vías, la entidad pública que presentó la tutela por medio de su oficina jurídica.',
        'Juzgados Quinto y Sexto Civiles de Circuito de Cartagena': 'Son los despachos judiciales contra los que se dirige la tutela; se toma la mención completa como organización.',
        '$22.586.296.153': 'Es el valor de la condena impuesta al instituto como indemnización; se etiqueta como monto (categoría adicional).',
        'octubre 7 de 2011': 'Es la fecha en que se dictó el fallo de primera instancia de la tutela.',
        'Tribunal Superior de Cartagena, Sala Civil Familia': 'Es el tribunal (y su sala) que dictó el fallo de primera instancia; spaCy marcó la sala como lugar y se corrigió.',
        '0075 de 2001': 'Es el número de radicación de un proceso judicial, un identificador relevante que no es persona, lugar ni fecha: MISC.',
        'Rafael Paternina Sumoza': 'Es la persona que promovió el proceso de radicación 0075 de 2001; se nombra con nombre y apellidos.',
        'Juzgado Quinto Civil del Circuito de Cartagena': 'Es el despacho judicial donde cursa el proceso; spaCy lo partió y marcó «Circuito de Cartagena» como lugar.',
        'Juez Quinto Civil del Circuito de Cartagena': 'Designa al titular del despacho que impugnó el fallo; no nombra a una persona, por eso se trata como la organización judicial.',
        'Corte Suprema de Justicia, Sala de Casación Civil': 'Es el tribunal de cierre de la jurisdicción ordinaria que revocó el fallo en segunda instancia.',
        'noviembre 11 de 2011': 'Es la fecha del fallo de segunda instancia que revocó la decisión.',
    },
}


def contexto(texto: str, ini: int, fin: int, ancho: int = 60) -> str:
    return '…' + texto[max(0, ini - ancho):fin + ancho].replace('\n', ' ') + '…'


def tabla_informe(nombre: str) -> pd.DataFrame:
    _, finales = revision[nombre]
    texto = fragmentos[nombre]
    filas = {}
    for e in finales:
        clave = (e['texto'], e['etiqueta'])
        if clave in filas:
            filas[clave]['Apariciones'] += 1; continue
        just = JUSTIFICACIONES.get(nombre, {}).get(e['texto'])
        if not just:
            just = f"⚠️ BORRADOR — {CATEGORIAS[e['etiqueta']]}; contexto: {contexto(texto, e['inicio'], e['fin'])}"
        filas[clave] = {'Entidad': e['texto'], 'Clasificación': e['etiqueta'], 'Justificación': just,
                        'Etiqueta de spaCy': e['etiqueta_spacy'], 'Origen': e['origen'], 'Apariciones': 1}
    return pd.DataFrame(filas.values())


tablas = {n: tabla_informe(n) for n in fragmentos}
pendientes = {n: int(t['Justificación'].str.startswith('⚠️').sum()) for n, t in tablas.items()}
print('Justificaciones pendientes (QA de la fase 5):', pendientes)

Justificaciones pendientes (QA de la fase 5): {'Fragmento 1': 0, 'Fragmento 2': 5}


> **Interpretación de la salida:** si hay justificaciones pendientes, son entidades del fragmento descargado que no estaban previstas en `JUSTIFICACIONES`. El borrador muestra su contexto: léelo, decide la categoría (ajusta `AJUSTES` si hace falta) y escribe la justificación en el diccionario. El informe está listo cuando el contador llega a 0 en ambos fragmentos.

### 5.1 Informe — Fragmento 1

In [11]:
f = FUENTES['Fragmento 1']
print(f"Fuente: {f['titulo']}\nEnlace: {f['url']}\nConsulta: {dt.date.today().isoformat()}\n")
print(fragmentos['Fragmento 1'])
tablas['Fragmento 1'][['Entidad', 'Clasificación', 'Justificación']]

Fuente: Corte Constitucional de Colombia. Sentencia SU-499 de 2024 (28 de noviembre de 2024)
Enlace: https://www.corteconstitucional.gov.co/relatoria/2024/su499-24.htm
Consulta: 2026-11-03

La Corte Constitucional amparó el derecho al debido proceso de la Universidad de Cartagena, vulnerado por la Sección Cuarta del Consejo de Estado al confirmar en segunda instancia mediante providencia del 11 de mayo de 2023, la sentencia de la Sala Fija de Decisión del Tribunal Administrativo de Bolívar proferida en primera instancia el 29 de mayo de 2020 dentro del proceso de nulidad y restablecimiento del derecho adelantado por Ecopetrol contra los actos administrativos mediante los cuales la Universidad de Cartagena le negó la devolución de los dineros pagados por concepto de “estampilla Universidad de Cartagena siempre a la altura de los tiempos”. En consecuencia, dejó sin efectos la sentencia de segunda instancia objeto de la tutela y ordenó a la Sección Cuarta del Consejo de Estado que rehicie

,Entidad,Clasificación,Justificación
0,Corte Constitucional,ORG,"Es el tribunal que dicta la sentencia y ampara el derecho; actúa como institución, no como lugar ni persona."
1,Universidad de Cartagena,ORG,Es la institución de educación superior cuyo derecho al debido proceso se ampara; «Cartagena» forma parte de su nomb...
2,Sección Cuarta del Consejo de Estado,ORG,Es la sección del máximo tribunal de lo contencioso administrativo que dictó la providencia cuestionada; spaCy la pa...
3,11 de mayo de 2023,DATE,Es la fecha de la providencia de segunda instancia que confirmó la sentencia; marca un momento del proceso.
4,Sala Fija de Decisión del Tribunal Administrativo de Bolívar,ORG,Es el órgano judicial que dictó la sentencia de primera instancia; «Bolívar» identifica su jurisdicción dentro del n...
5,29 de mayo de 2020,DATE,Es la fecha en que se dictó la sentencia de primera instancia.
6,nulidad y restablecimiento del derecho,MISC,"Nombra un medio de control judicial (tipo de proceso), no una persona ni una institución; por eso es MISC."
7,Ecopetrol,ORG,"Es la empresa que promovió el proceso contra los actos de la Universidad; actúa como parte, es una organización."
8,Universidad de Cartagena,LOC,Es la institución de educación superior cuyo derecho al debido proceso se ampara; «Cartagena» forma parte de su nomb...


### 5.2 Informe — Fragmento 2

In [12]:
f = FUENTES['Fragmento 2']
print(f"Fuente: {f['titulo']}\nEnlace: {f['url']}\nConsulta: {dt.date.today().isoformat()}\n")
print(fragmentos['Fragmento 2'])
tablas['Fragmento 2'][['Entidad', 'Clasificación', 'Justificación']]

Fuente: Corte Constitucional de Colombia. Sentencia T-390 de 2012 (28 de mayo de 2012), M. P. Nilson Pinilla Pinilla
Enlace: https://www.corteconstitucional.gov.co/relatoria/2012/t-390-12.htm
Consulta: 2026-11-03

Esclarecido el panorama normativo que determina cuál es la jurisdicción a cargo de las controversias y litigios originados en la actividad de las entidades públicas, pasa la Sala a decidir el objeto de estudio, ante la acción de tutela que el Jefe de la Oficina Jurídica de INVÍAS presentó contra los Juzgados Quinto y Sexto Civiles de Circuito de Cartagena, al estimar vulnerado el derecho al debido proceso al ser condenado ese Instituto a pagar $22.586.296.153, como indemnización por la ocupación de predios, a causa de la realización de obras públicas. 6.2. El fallo de primera instancia en la acción de tutela, dictado en octubre 7 de 2011 por el Tribunal Superior de Cartagena, Sala Civil Familia, amparó únicamente lo relacionado con el proceso de radicación 0075 de 2001, promo

,Entidad,Clasificación,Justificación
0,INVÍAS,ORG,"Es el Instituto Nacional de Vías, la entidad pública que presentó la tutela por medio de su oficina jurídica."
1,Juzgados Quinto y Sexto Civiles de Circuito de Cartagena,ORG,Son los despachos judiciales contra los que se dirige la tutela; se toma la mención completa como organización.
2,$22.586.296.153,MONEY,Es el valor de la condena impuesta al instituto como indemnización; se etiqueta como monto (categoría adicional).
3,octubre 7 de 2011,DATE,Es la fecha en que se dictó el fallo de primera instancia de la tutela.
4,"Tribunal Superior de Cartagena, Sala Civil Familia",ORG,Es el tribunal (y su sala) que dictó el fallo de primera instancia; spaCy marcó la sala como lugar y se corrigió.
5,0075 de 2001,MISC,"Es el número de radicación de un proceso judicial, un identificador relevante que no es persona, lugar ni fecha: MISC."
6,Rafael Paternina Sumoza,PER,Es la persona que promovió el proceso de radicación 0075 de 2001; se nombra con nombre y apellidos.
7,Juzgado Quinto Civil del Circuito de Cartagena,ORG,Es el despacho judicial donde cursa el proceso; spaCy lo partió y marcó «Circuito de Cartagena» como lugar.
8,Roberto Julio Romero,PER,"⚠️ BORRADOR — Persona; contexto: …rios, el de radicación 0098 de 2006, promovido por el señor Roberto Julio Romero [..."
9,Juzgado Sexto Civil,MISC,"⚠️ BORRADOR — Otra entidad relevante (eventos, productos, títulos, normas, procesos); contexto: …o por el señor Robe..."


> **Interpretación de la salida:** estas son las tablas que pide la actividad (*Entidad · Clasificación · Justificación*), una por fragmento. Para la trazabilidad, las columnas `Etiqueta de spaCy` y `Origen` de `tablas[...]` muestran qué propuso la herramienta y qué se cambió a mano.

### 5.3 Reflexión final

La reflexión (máximo 5 líneas) se apoya en las cifras de la fase 4. La celda siguiente las reúne; el texto debe redactarse con base en ellas.

In [13]:
m = medidas
tot_auto, tot_fin = int(m['Detectadas por spaCy'].sum()), int(m['Tras revisión'].sum())
correctas = int(m['Correctas sin cambios'].sum())
n_fechas = sum(e['etiqueta'] == 'DATE' for n in revision for e in revision[n][1])
print(f'spaCy detectó {tot_auto} entidades distintas; tras la revisión quedaron {tot_fin}, '
      f'de las cuales {correctas} estaban bien sin cambios ({correctas / tot_fin:.0%}).')
print(f'Las {n_fechas} fechas solo aparecieron gracias al EntityRuler: el modelo no tiene la categoría DATE.')

spaCy detectó 27 entidades distintas; tras la revisión quedaron 22, de las cuales 15 estaban bien sin cambios (68%).
Las 3 fechas solo aparecieron gracias al EntityRuler: el modelo no tiene la categoría DATE.


**Reflexión (ejemplo; ajústala a tus cifras):** la herramienta encontró la mayoría de las instituciones, pero no bastó por sí sola: partió nombres de tribunales (*Sección Cuarta* | *Consejo de Estado*), marcó como lugar o persona palabras en mayúscula al inicio de oración y tomó órganos judiciales por lugares. Las fechas solo se detectaron al añadir patrones, y el medio de control y el número de radicación se agregaron a mano como MISC. La revisión manual fue necesaria para que cada etiqueta correspondiera al sentido jurídico del texto.

---

## 6. 🔷 Fase 6 — Monitoreo y mantenimiento

**Aspecto específico:** el resultado depende del texto, del modelo y de su versión. Para repetir la actividad con otros fragmentos o para que otra persona reproduzca este informe:

| Cambio | Qué revisar | Dónde |
|---|---|---|
| Otros fragmentos (propios) | Cambiar `FUENTES` o `FRAGMENTOS_MANUALES`; vaciar `AJUSTES` y `JUSTIFICACIONES` y llenarlos tras ver la salida cruda | Fases 2, 4 y 5 |
| Otro dominio (salud, noticias, técnico) | Agregar patrones propios al EntityRuler (p. ej. normas, códigos) | Fase 3 |
| Otra versión de spaCy o del modelo | Las entidades detectadas pueden cambiar: volver a revisar los avisos de la fase 4 | Registro de la fase 3 |
| Textos con muchos errores repetidos | Considerar entrenar un NER propio | `training_ner.ipynb` |

**Lista de verificación antes de entregar:**
- [ ] Dos fragmentos reales, ≥ 150 palabras cada uno, con título, enlace y fecha de consulta.
- [ ] Librería, versión, modelo y parámetros declarados (fase 3).
- [ ] Ninguna justificación marcada como borrador (fase 5).
- [ ] Clasificaciones revisadas según el contexto, aunque spaCy sugiriera otra etiqueta.
- [ ] Reflexión de máximo 5 líneas apoyada en las cifras de la fase 4.

In [14]:
# Registro de versiones para reproducir el informe
import platform
pd.Series({'Python': platform.python_version(), **registro_herramienta,
           'Fecha de ejecución': dt.datetime.now().strftime('%Y-%m-%d %H:%M')}).to_frame('Valor')

,Valor
Python,3.14.6
Librería,spaCy 3.8.16
Modelo,es_core_news_md 3.8.0 (licencia: GNU GPL 3.0)
Componentes,"tok2vec, morphologizer, parser, attribute_ruler, lemmatizer, entity_ruler, ner"
Parámetros,"EntityRuler antes de ner con 6 patrones (DATE, MONEY); sin umbrales de confianza"
Fecha de ejecución,2026-11-03 21:17


> **Interpretación de la salida:** con esta tabla otra persona puede reproducir exactamente el informe. Si cambia la versión del modelo y aparecen avisos en la fase 4, es señal de que la salida automática cambió y la revisión debe repetirse.

## Referencias

- Corte Constitucional de Colombia. Sentencia SU-499 de 2024. https://www.corteconstitucional.gov.co/relatoria/2024/su499-24.htm
- Corte Constitucional de Colombia. Sentencia T-390 de 2012. https://www.corteconstitucional.gov.co/relatoria/2012/t-390-12.htm
- Honnibal, M. & Montani, I. spaCy: Industrial-strength Natural Language Processing in Python. https://spacy.io
- Studer, S. et al. (2021). Towards CRISP-ML(Q): A Machine Learning Process Model with Quality Assurance Methodology. *Machine Learning and Knowledge Extraction*, 3(2).
- Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 17.